# NMF: The Other Topic Model

**Computational Text Analysis Pack · Module 08 — Topic Modelling**

*Technique:* Non-negative matrix factorisation; comparing two models on one corpus

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/05-unsupervised-structure/08b_nmf_topics.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

NMF factorises the document–term matrix into two non-negative matrices:
documents × topics, and topics × words. No probability model, no Dirichlet prior,
no sampling.

Practical consequences that matter more than the mathematics:

- **Deterministic** for a given seed and initialisation — same input, same topics
- Works on **TF-IDF**, so common words are down-weighted for you
- Usually gives **crisper, more readable** topics on small corpora
- Gives you no principled way to compare models, which LDA at least attempts

Use both. Where they agree, you have something.

## The tooling

**scikit-learn**'s `LatentDirichletAllocation` and `NMF`. **gensim** is faster
for large corpora and **MALLET** (Java) is what most published DH topic-model
work actually used — worth knowing when you read the literature.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation

LANGUAGE = "English"        # ← change this
N_TOPICS = 8

sub = df[df["language"] == LANGUAGE].reset_index(drop=True)
tfidf = TfidfVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=5, max_df=0.5,
                        max_features=3000, sublinear_tf=True)
X = tfidf.fit_transform(sub["text"])
vocab = np.array(tfidf.get_feature_names_out())

nmf = NMF(n_components=N_TOPICS, random_state=0, init="nndsvd", max_iter=400)
W = nmf.fit_transform(X)
print(f"{N_TOPICS} NMF topics from {X.shape[0]:,} documents")
print()
for t, comp in enumerate(nmf.components_):
    print(f"topic {t}: {', '.join(vocab[np.argsort(comp)[::-1][:12]])}")

## Side by side

Fit LDA on the same documents and compare. Read across the two lists, not down
each one.

In [ ]:
cvec = CountVectorizer(token_pattern=r"[\w\u0300-\u036f]+", min_df=5, max_df=0.5,
                       max_features=3000)
Xc = cvec.fit_transform(sub["text"])
cvocab = np.array(cvec.get_feature_names_out())
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=0,
                                learning_method="batch", max_iter=25).fit(Xc)

print(f"{'#':<3}{'NMF (tf-idf)':<52}{'LDA (counts)'}")
print("-" * 104)
for t in range(N_TOPICS):
    a = ", ".join(vocab[np.argsort(nmf.components_[t])[::-1][:6]])
    b = ", ".join(cvocab[np.argsort(lda.components_[t])[::-1][:6]])
    print(f"{t:<3}{a:<52}{b}")
print()
print("Topic NUMBERS do not correspond across models — topic 3 in one has no")
print("relation to topic 3 in the other. Compare the sets of word lists.")

In [ ]:
# Match them properly: which NMF topic is closest to which LDA topic?
def normalise(M):
    return M / M.sum(axis=1, keepdims=True)

shared = sorted(set(vocab) & set(cvocab))
idx_n = [list(vocab).index(w) for w in shared]
idx_l = [list(cvocab).index(w) for w in shared]
A = normalise(nmf.components_[:, idx_n] + 1e-12)
B = normalise(lda.components_[:, idx_l] + 1e-12)
sim = A @ B.T / (np.linalg.norm(A, axis=1)[:, None] * np.linalg.norm(B, axis=1)[None, :])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim, cmap="viridis")
ax.set_xlabel("LDA topic"); ax.set_ylabel("NMF topic")
ax.set_title("Cosine similarity between the two models' topics")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()

best = sim.max(axis=1)
print(f"{(best > 0.5).sum()} of {N_TOPICS} NMF topics have a close LDA counterpart")
print("Those are the topics worth building an argument on.")

### 🔧 Your turn 1

Find one NMF topic with a strong LDA match and one with none.

Read the top documents for each. The unmatched one is usually either an artefact
(boilerplate, a single outlet's style) or a genuinely fine distinction only one
model picked up — and you can only tell by reading.

In [ ]:
# NMF is deterministic; check it yourself rather than taking my word.
w2 = NMF(n_components=N_TOPICS, random_state=99, init="nndsvd",
         max_iter=400).fit(X)
same = all(set(vocab[np.argsort(nmf.components_[t])[::-1][:10]])
           == set(vocab[np.argsort(w2.components_[t])[::-1][:10]])
           for t in range(N_TOPICS))
print(f"identical top-10 words across two different seeds: {same}")
print()
print("With init='nndsvd' the initialisation is deterministic, so the seed does")
print("not matter. Switch to init='random' and it will — the reproducibility is")
print("a property of the initialisation, not of NMF itself.")

### 🔧 Your turn 2

Run NMF on the four Nigerian languages separately, at the same `N_TOPICS`.

Do comparable topics emerge? If health reporting produces a recognisable topic in
all four, that is evidence the topics reflect subject matter rather than
language-specific quirks — a much stronger claim than any single-language model.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Unmatched topics are more often artefacts than discoveries. The
commonest cause is a small set of near-duplicate documents — a syndicated story
carried by several outlets — which both TF-IDF and NMF are good at isolating. Do
check for duplicates before interpreting any small, very crisp topic.

**Your turn 2.** Subject-matter topics do tend to recur across the four
languages, which is the validation the method usually lacks. Two cautions: the
categories are not sampled identically across languages, and topics defined by
distinctive named entities will not travel. Cross-language topic replication is
an underexplored technique and this corpus supports it directly.

</details>

## Where this stops

You can fit two topic models and compare them. Next: **08c**, on choosing the
number of topics and on the things topic models get credit for that they do not do.